In [ ]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from locallib.picarrodb import *
from locallib.box import *
from locallib.query import *
from locallib.pandas import *

import sqlite3
import os
import pandas as pd

/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


# Connect to the Italgas databaes

In [3]:
DATABASE_PATH = os.path.join(os.getcwd(), "database/" "italgas_g2g_anders.db")
# Create a connection to the Italgas database using sqlite3
conn = sqlite3.connect(DATABASE_PATH)
print(DATABASE_PATH)

/home/sandbox/personal-repos/DA-3590/database/italgas_g2g_anders.db


In [4]:
query = 'SELECT * FROM LEAKS'
read_df = pd.read_sql_query(query, conn)
print(len(read_df))

235681


In [5]:
read_df['UniqueIdentifier'] = read_df['lisa'].str.replace('-LISA', '-L-')

In [6]:
ES = EmissionSource
ES.delete_column('Lisa')
ES.columns
# Fix for MSSQL uniqueidentifier conversion error: only cast DB field, not temp table column
es_query = f"""SELECT {ES.get_columns()},
            (SELECT P.GpsLatitude FROM Peak P WHERE P.Id = ES.RepresentativePeakId) AS PeakLatitude,
            (SELECT P.GpsLongitude FROM Peak P WHERE P.Id = ES.RepresentativePeakId) AS PeakLongitude
            FROM EmissionSource ES 
            WHERE CAST(ES.UniqueIdentifier AS NVARCHAR(100)) IN (SELECT UniqueIdentifier FROM #TempLisa)"""
       

read_df.db.set_query(es_query)
read_df.db.execute([EU1_Conn, EU2_Conn], temp_table_name='#TempLisa', source_col='UniqueIdentifier', append=False)

ProgrammingError: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Conversion failed when converting from a character string to uniqueidentifier. (8169) (SQLExecute)')

In [ ]:
reports = get_reports(customer_name='Italgas',years =[2026]).execute([EU1_Conn, EU2_Conn])
reports.db.set_query(query_reports_view("temp_reports"))
reports_dh = reports.db.execute(DATAHUB_Conn,'ReportId','temp_reports', append = False)

In [ ]:
#Missing Surveyor
leak_columns = [
    "leakId", "numProgressivo", "lisa", "aereoInterrato", "codiceDispersione", "codStato", "xCoord", "yCoord",
    "statoFoglietta", "codValidazione", "statoValidazione", "intervento", "dataInserimento", "dataArrivoSulCampo",
    "dataLocalizzazione", "dataRiparazione", "cap", "comune", "indirizzo", "indirizzoLisa", "indirizzoLocalizzazione",
    "indirizzoRiparazione", "accertamentoRiscontrato", "descrizioneAsset", "sedeTecnicaLocalizzata", "dataUltimaMod",
    "picarroLastUpdated", "idAzienda", "reportId", "lisaId", "LeakFound", "BoundaryName", "Region", "City",
    "PCubedReportName", "PCubedReportGuid", "LastSurveyDate", "LastSurveyDateStr", "PCubedReportTitle",
    "PCubedReportDate", "PCubedReportDateStr", "AssetCoverageFrac", "PipelineMeters", "BoxId", "SurveyorUnitName",
    "PeakLatitude", "PeakLongitude", "PeakEpoch", "DrivingStatus", "Amplitude", "CH4", "AggregatedEthaneRatio",
    "AggregatedDisposition", "AggregatedClassificationConfidence", "PeakNumber", "EthaneRatioSdev", "Sigma",
    "PriorityScore", "EmissionRate", "EmissionRateUpperBound", "EmissionRateLowerBound", "UpdateTimeString"
]

renamed_cols = {
    "BoundaryRegion" : "City",
    "ReportName" : "PCubedReportName",
    "ReportId" : "PCubedReportGuid",
    "ReportTitle" : "PCubedReportTitle",
    "ReportDate" : "PCubedReportDate",
    "EthaneRatio" : "AggregatedEthaneRatio",
    "Disposition" : "AggregatedDisposition",
    "ClassificationConfidence" : "AggregatedClassificationConfidence",
    "ReportAssetLengthKm" : "PipelineMeters"
}

In [ ]:

es_query = f"""SELECT {ES.get_columns()},
            (SELECT P.GpsLatitude FROM Peak P WHERE P.Id = ES.RepresentativePeakId) AS PeakLatitude,
            (SELECT P.GpsLongitude FROM Peak P WHERE P.Id = ES.RepresentativePeakId) AS PeakLongitude
            FROM EmissionSource ES WHERE ReportId IN (SELECT ReportId FROM #TempReports)
            AND ES.IsFiltered = 0"""

In [ ]:
reports.db.set_query(es_query)
emission_sources = reports.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReports', source_col = 'ReportId', append = False)
print(len(emission_sources))

In [ ]:
merged_leaks = pd.merge(read_df, emission_sources, on = 'UniqueIdentifier', how = 'right')

In [ ]:
print(len(merged_leaks))

In [ ]:
# Create LeakFound column based on G2G criteria
merged_leaks['LeakFound'] = ''
merged_leaks.loc[merged_leaks['codiceDispersione'].isin(['A1', 'A2', 'B', 'C', 'PRELOCALIZZATA']), 'LeakFound'] = 'Found_Gas_Leak'
merged_leaks.loc[merged_leaks['codiceDispersione'] == '', 'LeakFound'] = 'Not_Investigated'
merged_leaks.loc[merged_leaks['codiceDispersione'] == '', 'codiceDispersione'] = 'No Grade'
merged_leaks.loc[merged_leaks['aereoInterrato'] == '', 'aereoInterrato'] = 'NC'
merged_leaks.loc[merged_leaks['intervento'].isin(['INTERESSA IMPIANTI ALTRI SERVIZI-RP',
                                                    'INTERESSA IMPIANTO ALTRO DISTRIBUTORE-RP']), 'LeakFound'] = 'Found_Other_Source'
merged_leaks.loc[merged_leaks['intervento'].isin(['COMUNE INDIRIZZO ERRATO-RP', 'ANOMALIA NON RISCONTRATA-RP']),
                    'LeakFound'] = 'No_Gas_Found'
